In [30]:
'''
            UŻYCIE AI

Kod został napisany z wykorzystaniem 
generatywnej AI. Autorzy zweryfikowali
wszystko i prezentują własny wkład intelektualny,
biorąc pełną odpowiedzialność za ostateczną postać pracy
'''

'\n            UŻYCIE AI\n\nKod został napisany z wykorzystaniem \ngeneratywnej AI. Autorzy zweryfikowali\nwszystko i prezentują własny wkład intelektualny,\nbiorąc pełną odpowiedzialność za ostateczną postać pracy\n'

In [79]:
import numpy as np
import pandas as pd

In [80]:
# wczytanie lookupu stref
zones = pd.read_csv("taxi_zone_lookup.csv")

# zostawiamy tylko strefy z Manhattanu
manhattan_zones = zones.loc[zones["Borough"] == "Manhattan", "LocationID"].tolist()

manhattan_zones;

In [81]:
''' 
    WCZYTANIE FHV. FITLROWANIE. 
    
    Wyrzucamy kolumny "dispatching_base_num",
        "originating_base_num", "driver_pay".
    Zostawiamy tylko te przejazdy, które:
    - wykonywał Uber
    - zaczynają i kończą się na Manhattanie. 
    - są zwykłe (NIE wav, shared, access)
    - mają dodatnie / nieujemne wartości w 
        polach gdzie inne nie mają sensu (trip_miles,
        trip_time, base_passenger_fare, tolls, bcf, sales_tax,
        tips)
    - mają congestion_surcharge zgodne z
        https://www.nyc.gov/site/tlc/about/congestion-surcharge.page
        (0 lub 2.75)
    - mają cbd_congestion_fee zgodne z
        https://congestionreliefzone.mta.info/tolling
        (0 lub 1.5)
    - mają zerowe wartości airport_fee
'''

fhv = (pd.read_parquet("fhvhv_tripdata_2025-03.parquet")
      .drop(columns = ["dispatching_base_num", "originating_base_num",
                       "driver_pay"])
      .query('hvfhs_license_num == "HV0003" and '
             'PULocationID in @manhattan_zones and '
             'DOLocationID in @manhattan_zones and '
             'wav_request_flag == "N" and '
             'shared_request_flag == "N" and '
             'shared_match_flag == "N" and '
             'access_a_ride_flag == "N" and '
             'trip_miles > 0 and '
             'trip_time > 0 and '
             'base_passenger_fare > 0 and '
             'tolls >= 0 and '
             'bcf >= 0 and '
             'sales_tax >= 0 and '
             'tips >= 0 and '
             'congestion_surcharge in (0, 2.75) and '
             'cbd_congestion_fee in (0, 1.5) and '
             'airport_fee == 0') 
      )

In [82]:
# Sprawdzamy czy są jakieś brakujące wartości
fhv.isna().values.any()

np.False_

In [83]:
# Wyciągamy z pickup_datetime godzinę i dzień tygodnia
fhv = fhv.assign(hour = fhv["pickup_datetime"].dt.hour,
                 weekday = fhv["pickup_datetime"].dt.weekday,
                 month = fhv["pickup_datetime"].dt.month)
# weekday 0 to poniedziałek

In [84]:
# Sprawdzamy czy jest poprawny miesiąc
fhv['month'].value_counts()

month
3    3491641
Name: count, dtype: int64

In [85]:
# Dodajemy kolumnę z opłatą całkowitą
fhv["total_amount"] = (fhv["base_passenger_fare"] + fhv["tolls"] +
                       fhv["bcf"] + fhv["sales_tax"] + 
                       fhv["congestion_surcharge"] +
                       fhv["cbd_congestion_fee"])

In [86]:
# Sprawdzamy czy dobrze się dodało
(fhv["total_amount"] < fhv["base_passenger_fare"]).sum()

np.int64(0)

In [87]:
# Zmieniamy czas na minuty zamiast sekund
fhv["trip_time_mins"] = fhv["trip_time"]/60

In [88]:
# Usuwamy kolumny niepotrzebne do modelu
fhv = fhv.drop(columns = ["on_scene_datetime", "tolls", "bcf",
                          "sales_tax", "congestion_surcharge",
                          "airport_fee", "tips", "shared_request_flag",
                          "shared_match_flag", "access_a_ride_flag",
                          "wav_request_flag", "wav_match_flag", 
                          "cbd_congestion_fee", "request_datetime",
                          "dropoff_datetime",   #"pickup_datetime",
                          "PULocationID", "DOLocationID", "trip_time"]);

In [89]:
fhv

,hvfhs_license_num,pickup_datetime,trip_miles,base_passenger_fare,hour,weekday,month,total_amount,trip_time_mins
18,HV0003,2025-03-01 00:17:10,1.47,23.83,0,5,3,30.75,9.983333
19,HV0003,2025-03-01 00:33:19,3.23,18.05,0,5,3,24.33,12.000000
24,HV0003,2025-03-01 00:35:45,4.68,41.67,0,5,3,50.91,22.183333
35,HV0003,2025-03-01 00:45:35,2.90,59.79,0,5,3,70.79,15.650000
39,HV0003,2025-03-01 00:26:12,3.04,21.92,0,5,3,24.58,18.066667
...,...,...,...,...,...,...,...,...,...
20536859,HV0003,2025-03-31 23:09:12,3.97,20.30,23,0,3,22.66,16.716667
20536860,HV0003,2025-03-31 23:37:17,3.98,17.00,23,0,3,18.93,13.733333
20536863,HV0003,2025-03-31 23:01:02,2.30,52.25,23,0,3,62.78,12.566667
20536866,HV0003,2025-03-31 23:54:41,1.52,15.96,23,0,3,21.97,7.866667


In [90]:
# Eksportujemy fhv do csv
fhv.to_csv("uber_do_statOp.csv", index = False)